# STPBench Demo

**Dataset:** `ncche/xenium` — NCCHE Lung Adenocarcinoma (Xenium)  
**Example models:** `StNet`, `BLEEP`, `EGN`

Cells default to `dry_run=True` so the notebook is safe to run without a GPU.  
Set `DRY_RUN = False` when you are ready to launch preprocessing and training.

## 0. Setup

In [ ]:
from stpbench import STPred

# ── Configuration ─────────────────────────────────────
REPO_ROOT     = ".."  
DATA          = "ncche/xenium"
EXTERNAL_DATA = "hest/LUAD"      # dataset used for external evaluation
MODELS        = ["StNet", "EGN", "BLEEP", "TRIPLEX"]
GPU           = 1
DRY_RUN       = True             # set False to run actual jobs
# ──────────────────────────────────────────────────────

## 1. Initialize STPred

In [ ]:
stp = STPred(
    repo_root=REPO_ROOT,
    models=MODELS,
    gpu=GPU,
    dry_run=DRY_RUN,
    log_file="logs/demo.jsonl",
)

## 2. Discover Available Configs

In [ ]:
# Use instance methods — repo_root is inferred from stp automatically
print("Data configs :", stp.list_data())
print("Model configs:", stp.list_models())

In [ ]:
stp.describe_data(DATA)

## 3. Preflight Check

In [ ]:
check = stp.check(data=DATA, strict=False)
print("OK:", check["ok"])
if check["missing"]:
    print(f"Missing artifacts ({len(check['missing'])}):")
    for p in check["missing"][:5]:
        print(" ", p)

## 4. Preprocess

Generates patches, ST expression, gene sets, CV splits, and patch embeddings.  
Already-completed steps are skipped automatically.

In [ ]:
stp.preprocess(data=DATA)

## 5. Train

In [ ]:
train_result = stp.train(data=DATA)
train_result

## 6. Evaluate

In [ ]:
# Internal evaluation: PCC on cross-validation test folds
int_result = stp.evaluate_internal(data=DATA)
int_result.summary()

In [ ]:
# External evaluation: apply the trained model to a held-out dataset
ext_result = stp.evaluate_external(data=EXTERNAL_DATA, train_data=DATA)
ext_result.summary()

## 7. Inspect Results

`BenchmarkResult` is dict-compatible and provides helpers for summaries, DataFrames, checkpoints, and CSV export.

In [ ]:
int_result.to_dataframe()

In [ ]:
# Best checkpoint paths per model/fold
int_result.best_checkpoints()

In [ ]:
int_result.save("../output/demo_results.csv")
print("Saved to output/demo_results.csv")

## 8. Resume From a Known Run

If training already finished, reload the run by timestamp and re-run evaluation without retraining.

In [ ]:
# Find the timestamp in logs/ then uncomment below

# TIMESTAMP = "2026-05-28-12-00-00"

# resumed = STPred.from_run(
#     data=DATA,
#     models=MODELS,
#     timestamp=TIMESTAMP,
#     gpu=GPU,
# )
# resumed.evaluate_external(data=EXTERNAL_DATA).summary()

## 9. One-Shot Benchmark (Optional)

Runs preprocess → train → internal eval → external eval in a single call.

In [ ]:
# stp2 = STPred(models=MODELS, gpu=GPU)
# result = stp2.benchmark(internal_data=DATA, external_data=EXTERNAL_DATA)
# result.summary()